# BaseDyGraph — W&B sweeps

Companion to `BaseDyGraph_experiments.ipynb` (the ablation ladder). This notebook runs
**hyperparameter sweeps** driven by `configs/sweep.yaml`, over the base model/data in
`configs/model.yaml` and `configs/data.yaml`.

How it works: each sweep trial gets a set of overrides from W&B. The `train()` function
routes every override to the right config by inspecting the dataclass fields —
`SyntheticGraphDataConfig` fields regenerate the dataset (cached so identical data is reused),
`ModelConfig` fields rebuild the model, and `lr`/`weight_decay`/`max_epochs`/`seed` set training.
The sweep metric is `val/acc` (logged each epoch); a composite `final/sweep_score` is also logged.

In [ ]:
import sys, pathlib
# notebooks live in notebooks/; modules live in ../src; configs in ../configs
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
CONFIG_DIR = ROOT / 'configs'

import copy, json
import yaml
import torch
import lightning.pytorch as pl
import wandb

from synthetic_generators import SyntheticGraphDataConfig, build_generator
from data_module import DiscreteStateDataModule
from model import DiscreteSTGraphLightningModule
from utilities import ModelConfig
from evaluation_utilities import full_evaluation_report

In [ ]:
wandb.login()   # prompts for an API key the first time; no-op if already logged in

## Load the base configs

Edit the YAML files in `configs/` rather than this notebook. `model.yaml` carries the base
architecture + a `training:` block; `data.yaml` carries the generator settings, splits and
batch size; `sweep.yaml` is the W&B sweep spec.

In [ ]:
def _load(name):
    with open(CONFIG_DIR / name) as f:
        return yaml.safe_load(f)

DATA_YAML  = _load('data.yaml')
MODEL_YAML = _load('model.yaml')
SWEEP_CONFIG = _load('sweep.yaml')

BASE_DATA   = DATA_YAML['data']
BASE_SPLITS = DATA_YAML['splits']
BASE_BATCH  = int(DATA_YAML['batch_size'])
BASE_MODEL  = MODEL_YAML['model']
BASE_TRAIN  = MODEL_YAML['training']

print('sweep method :', SWEEP_CONFIG.get('method'))
print('sweep metric :', SWEEP_CONFIG.get('metric'))
print('sweeping     :', sorted(SWEEP_CONFIG['parameters'].keys()))

## Routing + data cache

`route_overrides` sends each swept field to data / model / training by membership in the
dataclass field sets. `build_data` memoises generation keyed by the resolved data config, so a
sweep over *model* hyperparameters generates the dataset once and reuses it across trials.

In [ ]:
DATA_FIELDS  = set(SyntheticGraphDataConfig.__dataclass_fields__)
MODEL_FIELDS = set(ModelConfig.__dataclass_fields__)
TRAIN_FIELDS = {'lr', 'weight_decay', 'max_epochs', 'seed'}

_DATA_CACHE = {}   # persists across trials in this process

def build_data(data_cfg: dict, splits: dict, batch_size: int):
    """Generate the dataset once per distinct data config; return (datamodule, regime_graphs)."""
    key = json.dumps({'data': data_cfg, 'splits': splits, 'bs': batch_size}, sort_keys=True)
    if key in _DATA_CACHE:
        return _DATA_CACHE[key]
    synth = SyntheticGraphDataConfig(**data_cfg)
    gen = build_generator(synth, lead_lag=False)        # ContemporaneousGraphGenerator
    data = gen.generate()
    s, r = data['state_ids'], data['regimes']
    n = s.size(0)
    assert splits['train'] + splits['val'] + splits['test'] == n, 'splits must sum to num_samples'
    a, b = splits['train'], splits['train'] + splits['val']
    dm = DiscreteStateDataModule(
        train_tensor=s[:a], val_tensor=s[a:b], test_tensor=s[b:],
        train_regimes=r[:a], val_regimes=r[a:b], test_regimes=r[b:],
        batch_size=batch_size, num_workers=0,
    )
    dm.setup()
    out = (dm, data['regime_graphs'])
    _DATA_CACHE[key] = out
    return out

def route_overrides(overrides: dict):
    """Split a flat dict of W&B overrides into (data, model, training, batch_size)."""
    data_cfg  = copy.deepcopy(BASE_DATA)
    model_cfg = copy.deepcopy(BASE_MODEL)
    train_cfg = copy.deepcopy(BASE_TRAIN)
    batch_size = BASE_BATCH
    for k, v in overrides.items():
        if k == 'batch_size':      batch_size = int(v)
        elif k in TRAIN_FIELDS:    train_cfg[k] = v
        elif k in DATA_FIELDS:     data_cfg[k] = v
        elif k in MODEL_FIELDS:    model_cfg[k] = v
        # silently ignore anything else (e.g. W&B internals)
    return data_cfg, model_cfg, train_cfg, batch_size

def sweep_score(report: dict) -> float:
    """Composite objective: next-state accuracy + small bonuses for graph fidelity."""
    acc   = report.get('model_acc', 0.0)
    corr  = report.get('graph_corr', 0.0) or 0.0
    auroc = report.get('graph_auroc', 0.0) or 0.0
    return acc + 0.1 * max(0.0, corr) + 0.1 * max(0.0, auroc - 0.5)

## The trial function

Called once per sweep trial. `wandb.init()` (no args) picks up the trial's config from the agent.

In [ ]:
def train(config=None):
    run = wandb.init(config=config)        # agent injects the trial's hyperparameters
    try:
        overrides = dict(wandb.config)
        data_cfg, model_cfg, train_cfg, batch_size = route_overrides(overrides)

        # model dims are derived from the data so they always agree
        model_cfg['num_states'] = data_cfg['num_states']
        model_cfg['num_nodes']  = data_cfg['num_nodes']
        model_cfg['max_seq_len'] = max(int(model_cfg.get('max_seq_len', 512)), int(data_cfg['seq_len']))

        seed = int(train_cfg['seed'])
        max_epochs = int(train_cfg['max_epochs'])
        pl.seed_everything(seed, workers=True)

        dm, regime_graphs = build_data(data_cfg, BASE_SPLITS, batch_size)
        cfg = ModelConfig(**model_cfg)
        model = DiscreteSTGraphLightningModule(
            cfg, lr=float(train_cfg['lr']), weight_decay=float(train_cfg['weight_decay']),
            true_regime_graphs=regime_graphs, scheduler_t_max=max_epochs,
        )

        from lightning.pytorch.loggers import WandbLogger
        logger = WandbLogger(experiment=run)   # attach Lightning logs to this sweep run
        trainer = pl.Trainer(
            max_epochs=max_epochs, accelerator='auto', devices='auto',
            logger=logger, enable_checkpointing=False, enable_progress_bar=False,
            num_sanity_val_steps=0, log_every_n_steps=10,
        )
        trainer.fit(model, datamodule=dm)

        report = full_evaluation_report(
            model, dm.val_dataloader(), num_states=int(data_cfg['num_states']),
            true_regime_graphs=regime_graphs,
        )
        report['sweep_score'] = sweep_score(report)
        wandb.log({f'final/{k}': v for k, v in report.items() if isinstance(v, (int, float))})
    finally:
        wandb.finish()

## Launch the sweep

Set the project, then create a sweep and start an agent. To add more trials later (or run agents
on other machines in parallel), set `EXISTING_SWEEP_ID` to the id printed here and re-run the cell.

> **Note on early termination:** `sweep.yaml` enables Hyperband. Run from this notebook, the agent
> may not interrupt a trial mid-`fit` (Lightning does not poll W&B for a stop signal). For reliable
> pruning, use the CLI agent (one process per trial) described in `configs/sweep.yaml`.

In [ ]:
SWEEP_PROJECT = 'BaseDyGraph Sweep'
SWEEP_COUNT = 20            # trials this agent will run
EXISTING_SWEEP_ID = None   # set to a sweep id (str) to attach to an existing sweep

sweep_id = EXISTING_SWEEP_ID or wandb.sweep(SWEEP_CONFIG, project=SWEEP_PROJECT)
print('sweep id:', sweep_id)
print('dashboard: https://wandb.ai (project:', SWEEP_PROJECT, ')')

wandb.agent(sweep_id, function=train, count=SWEEP_COUNT)